# Phase-Plane Widget for the Mean-Field (FS + RS) Model

This notebook demonstrates the interactive phase-plane widget for the 2-population mean-field model.

The widget renders:
- **Phase plane** with nullclines, vector field, fixed points, and trajectories
- **Time series** for each population's firing rate

All computations (nullclines, vector field, fixed points, trajectories) are performed in Python using the exact `MFModel.rhs()` pipeline, while the widget UI handles Canvas rendering and interactivity.

Use the **slider controls** to vary external input rates, the population time constant `τ_f`, or the transfer-function scaling factors `α_FS` and `α_RS`.

The built-in **Export SVG** button generates a vector figure of the current phase-plane view.

This file was origninally written by Marmaduke Woodman (marmaduke.woodman@univ-amu.fr) and then modified by Ilaria Carannante (ilariac@kth.se)

In [1]:
import sys
sys.version

'3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]'

In [2]:
import sys
import os

# Adding the project folder to sys.path
cwd = os.getcwd()
parent_dir = os.path.abspath(os.path.join(cwd, '..', '..',))
sys.path.append(parent_dir)
# We do this so that we can directly import files in the utils folder

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import json
from ntmf.phase_plane_widget import PhasePlaneWidget
from ntmf.phase_plane import NTMFMeanField
from ntmf.config import get_network_config, get_params_model_SI, adding_K_params
from ntmf.validation import build_mf_network_config

In [4]:
# ── Build network config with external-input quantal conductances ──
network_config = build_mf_network_config(
    '../../config/network_config_file_MF_phase_plane.json',
    Q_e_nS=1.5, Q_i_nS=5.0,
)

# ── Neuron model parameters (SI units) ──
params_SI = {
    'FS': adding_K_params(
        get_params_model_SI('FS', '../../neuron_models/AdEx/FS.json').copy(),
        network_config,
    ),
    'RS': adding_K_params(
        get_params_model_SI('RS', '../../neuron_models/AdEx/RS.json').copy(),
        network_config,
    ),
}

# ── Load transfer-function fit parameters ──
with open('../../transfer_function/fitting/results/10_best_params_TF_FS.json') as f:
    fs_fits = json.load(f)
with open('../../transfer_function/fitting/results/10_best_params_TF_RS.json') as f:
    rs_fits = json.load(f)

fs_fit = fs_fits[0]   # first FS fit
rs_fit = rs_fits[0]   # first RS fit

poly_params = {
    'FS': fs_fit['polynomial_params'],
    'RS': rs_fit['polynomial_params'],
}
alphas = {
    'FS': fs_fit['alpha'],
    'RS': rs_fit['alpha'],
}

print(f"FS fit:  alpha = {alphas['FS']:.3f},  mean_error = {fs_fit.get('mean_error', 'n/a')}")
print(f"RS fit:  alpha = {alphas['RS']:.3f},  mean_error = {rs_fit.get('mean_error', 'n/a')}")

FS fit:  alpha = 1.167,  mean_error = 6.3979593489852205
RS fit:  alpha = 1.448,  mean_error = 0.15564526631361159


In [5]:
network_config

{'metainfo': 'Network config file -- MF phase plane',
 'network_composition': {'tot_neurons': 10000,
  'FS_neuron': 2000,
  'RS_neuron': 8000,
  'conn_prob': 0.05},
 'external_input': {'N_external_exc': 8000,
  'N_external_inh': 2000,
  'conn_prob': 0.05,
  'Q_e': 1.5,
  'Q_i': 5.0},
 'rates': {'freq_exc_FS': 3,
  'freq_exc_RS': 4,
  'freq_inh_FS': 1,
  'freq_inh_RS': 2,
  'background_freq': 0.0},
 'units': {'freq_exc_FS': 'Hz',
  'freq_exc_RS': 'Hz',
  'freq_inh_FS': 'Hz',
  'freq_ing_RS': 'Hz',
  'background_freq': 'Hz'}}

In [6]:
params_SI

{'FS': {'C_m': 2.0000000000000003e-10,
  'g_L': 1e-08,
  'E_L': -0.065,
  'a': 0.0,
  'b': 0.0,
  'tau_w': 0.001,
  'V_th': -0.048,
  'Delta_T': 0.0005,
  'V_reset': -0.065,
  'V_peak': 0.0,
  't_ref': 0.005,
  'E_e': 0.0,
  'Q_e': 1.5000000000000002e-09,
  'E_i': -0.08,
  'Q_i': 5e-09,
  'tau_syn': 0.005,
  'K_e': 400.0,
  'K_i': 100.0},
 'RS': {'C_m': 2.0000000000000003e-10,
  'g_L': 1e-08,
  'E_L': -0.065,
  'a': 0.0,
  'b': 1.0000000000000002e-10,
  'tau_w': 0.5,
  'V_th': -0.05,
  'Delta_T': 0.002,
  'V_reset': -0.065,
  'V_peak': 0.0,
  't_ref': 0.005,
  'E_e': 0.0,
  'Q_e': 1.5000000000000002e-09,
  'E_i': -0.08,
  'Q_i': 5e-09,
  'tau_syn': 0.005,
  'K_e': 400.0,
  'K_i': 100.0}}

In [7]:
# ── Build the MF → BaseModel adapter ──
mf_model = NTMFMeanField(
    params=params_SI,
    poly_params=poly_params,
    alphas=alphas,
    network_config=network_config,
    tau_f=0.01,
)

# ── Create the widget in Python-compute mode ──
widget = PhasePlaneWidget(
    model=mf_model,
    python_compute=True,
    xlim=[-5, 100],
    ylim=[-5, 100],
    t_max=10.0,
)

# The widget's initial computation is triggered automatically on display.

In [8]:
widget

## How it works

This widget is running in **Python-compute mode** (`python_compute=True`). Here's what happens on every slider change:

1. The JavaScript front-end detects the parameter change and syncs it to the Python kernel via `anywidget`.
2. Python's `PhasePlaneWidget._run_python_compute()` calls `BaseModel.compute_nullclines()`, `find_fixed_points()`, `compute_vector_field()`, and `compute_trajectory()` on the wrapped `NTMFMeanField` model.
3. These results are written to synced traitlets (e.g. `nullcline_x`, `fixed_points`, `trajectory`).
4. The traits sync back to the browser, and the JavaScript renderer repaints the Canvas with the fresh data.

### Key advantage
The `NTMFMeanField.f()` method calls the **exact same** `MFModel.rhs()` and `TF_template_sim()` functions used in production simulations. There is no approximation or model duplication.

### Exporting figures
Use the **Export SVG** button embedded in the widget to download a publication-quality vector rendering of the current phase plane (nullclines + vector field + fixed points + trajectory).